In [1]:
# ############################################################
# Level 3 — 실제 마트 장바구니 9835건 (공개 데이터)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기
# ------------------------------------------------------------
import pandas as pd
import urllib.request                             # URL에서 텍스트 파일 읽기
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
# ------------------------------------------------------------
# [목적] 데이터 불러오기 — 한 줄 = 한 번의 장바구니 (품목이 콤마로 나열)
# ------------------------------------------------------------
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv'
lines = urllib.request.urlopen(url).read().decode('utf-8').strip().splitlines()  # 줄 단위로
data = [ln.split(',') for ln in lines]            # 각 줄을 품목 리스트로 (장바구니 9835개)
te = TransactionEncoder()
df = pd.DataFrame(te.fit(data).transform(data), columns=te.columns_)   # 0/1 표

In [3]:
# ------------------------------------------------------------
# [목적] 빈발 조합(2% 이상) -> 신뢰도 0.3 이상 규칙 -> 향상도 순
# ------------------------------------------------------------
freq = apriori(df, min_support=0.02, use_colnames=True)                # 2% 이상 등장
rules = association_rules(freq, metric='confidence', min_threshold=0.3) # 신뢰도 30% 이상
rules = rules.sort_values('lift', ascending=False)                     # 강한 규칙부터
print(rules[['antecedents','consequents','support','confidence','lift']].head().to_string())

                                  antecedents                    consequents   support  confidence      lift
32  frozenset({whole milk, other vegetables})   frozenset({root vegetables})  0.023183    0.309783  2.842082
33   frozenset({whole milk, root vegetables})  frozenset({other vegetables})  0.023183    0.474012  2.449770
17               frozenset({root vegetables})  frozenset({other vegetables})  0.047382    0.434701  2.246605
19            frozenset({whipped/sour cream})  frozenset({other vegetables})  0.028876    0.402837  2.081924
35            frozenset({whole milk, yogurt})  frozenset({other vegetables})  0.022267    0.397459  2.054131


In [4]:
# ============================================================
# [결과 해석]
#  · 실제 거래 9835건에서 약 37개 규칙이 나옴
#  · 가장 강한 규칙: {other vegetables, whole milk} -> {root vegetables}, 향상도 ~ 2.84
#      -> 채소·우유를 산 사람은 뿌리채소를 우연보다 약 2.8배 자주 함께 삼
#  · 이런 규칙이 '함께 진열 / 묶음 할인 / 추천'의 근거가 됨 (실전 장바구니 분석)
# ============================================================